Load libraries and functions

In [1]:
import torch, pyro, pandas as pd
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from codebase_0 import set_seed, DeviceLoader
from poisson_horseshoe import HorseshoePoissonBCANN
from poisson_bcann_vi import poisson_fit_glm_mle, build_guide_horseshoe_bcann, poisson_bcann_svi_training_loop
from poisson_bcann_hmc_nuts import (
    get_device, to_device, stratified_subsample, make_chain_inits, time_nuts, run_nuts,
    nuts_diagnostics, nuts_posterior_stats, svi_posterior_stats, deviance_from_stats,
    width_by_bin, _rhat_ess,
)
from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, to_device
)

from poisson_nuts_parallel import run_nuts_parallel

seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perform train/validation/test split

In [ ]:
device = torch.device("cuda:0")

dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
)

Subsample dataset for HMC run

In [9]:
sub_ids = stratified_subsample(y_learn, 30000, seed = seed)
X_sub, y_sub, logexp_sub = X_learn[sub_ids], y_learn[sub_ids], torch.log(exp_learn[sub_ids])
sub_loader = DeviceLoader(
    X_sub, y_sub, logexp_sub,
    batch_size = 1024, # much smaller data set so we set minibatch so smaller
    shuffle= True
)

Initialise HorseshoePoissonBCANN, VI posterior (as guide) and create a chain object for NUTS

In [10]:
network_architecture = [20,15, 10]

model = HorseshoePoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh},
).to(device)
glm_mle = poisson_fit_glm_mle(
    X_sub, y_sub, 
    log_exposure=logexp_sub, random_state=seed
)
guide = build_guide_horseshoe_bcann(
    model, X_sub[:256], y_sub[:256], glm_mle=glm_mle, rank=10
)

bcann_history = poisson_bcann_svi_training_loop(
    model, 
    guide, 
    sub_loader, 
    val_loader,
    n_epochs=300, 
    lr=5e-3, 
    val_every=5, 
    patience=10, 
    random_state=seed, 
    device = device
)

inits = make_chain_inits(model, X_sub, y_sub, logexp_sub, num_chains=6, glm_mle=glm_mle,
                         guide=guide, how="svi_sample", seed=seed, device = device)


Epoch 1 | -ELBO/n: 0.2506 | val deviance: 0.3131
Epoch 25 | -ELBO/n: 0.2288 | val deviance: 0.3165
Epoch 50 | -ELBO/n: 0.2219 | val deviance: 0.3174
Early stopping at epoch 50. Best val deviance 0.3131 at epoch 1


Test run NUTS to see how long each iteration takes

In [11]:
time_nuts(
    model,
    X_sub,
    y_sub,
    logexp_sub,
    inits,
    n_iter = 50,
    device = device,
    jit_compile = True
)

1.93 s / iteration on cuda:0  (early warmup iterations are not representative of tree depth)


1.9334338521957397

Run NUTS with full config

In [12]:
WARMUP, SAMPLES = 1500, 1500
TARGET_ACCEPT = 0.9
MAX_TREE_DEPTH = 8

horse_shoe_samples, horse_shoe_info = run_nuts_parallel(
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ), 
    X_sub, 
    y_sub, 
    logexp_sub,
    inits,
    model_cls = HorseshoePoissonBCANN,
    out_dir = "poisson_nuts_chains_horseshoe",
    num_warmup=WARMUP, 
    num_samples=SAMPLES,
    target_accept=TARGET_ACCEPT, 
    max_tree_depth=MAX_TREE_DEPTH, 
    seed=seed, 
    device = device,
    progress_every = 5,
    jit_compile = True
)

elapsed 3:28:09
chain 5: Sample: 100%|██████████| 3000/3000 [3:24:08,  4.08s/it, step size=6.39e-04, acc. prob=0.961]


Write to disk for later uses

In [13]:
torch.save(
    {
        "samples": horse_shoe_samples,
        "info": horse_shoe_info,
        "config": dict(num_warmup=1000, num_samples=1000, target_accept=0.8,
                       max_tree_depth=10, seed=seed, n_chains=6),
    },
    "hmc_results/poisson_horseshoe_hmc_results.pt",
)
